# Study 913 — Tracking-Difference Persistence 🧾

**Two funds track the same index. One quietly returns a few basis points more
each year. Is last year's winner next year's winner?**

The fund-picker's folklore says yes: look up last year's **tracking difference** — the gap
between a fund's total return and its index's — and buy whoever won it, because good
tracking is a skill and skills persist. The duller rival rule says just buy the lowest
**expense ratio**, which is published in advance and costs nothing to read.

We test both on two families of daily **total-return** closes, as-of 2026-06-30:

- **S&P 500 ETF trio** — SPY (9.45 bp), IVV (3 bp), VOO (3 bp): 2010-09-09 →
  2026-06-30, 3,975 days, 15 complete calendar years
- **S&P 500 full ladder** — the trio plus three NAV-priced index mutual funds
  (VFIAX 4 bp, FXAIX 1.5 bp, SWPPX 2 bp), which widen the fee ladder
- **Nasdaq-100 pair** — QQQ (20 bp) and QQQM (15 bp), 5 complete years

*Numbers below are the frozen headline run (`docs/results.md`, fingerprints
`f11699949902` / `82068f45e436` / `b33e0709c240`); the only live cells run the offline
synthetic control and say so. **SPLG was requested and is unavailable** — Yahoo! Finance
serves a single stale bar for it, so it is declared missing rather than quietly swapped.*


## 1. What a tracking difference actually is

An index fund promises to hand you the index. It never quite does. The fee comes out, dividends get reinvested a day late, the fund trades to keep up when the index reshuffles, and some of the shortfall comes back as securities-lending income. The annual leftover is the **tracking difference** — usually a handful of basis points, where a basis point is one hundredth of one percent.

So: does the fund that tracked best last year track best again? Here is the real table for the three big S&P 500 ETFs, each year's returns measured against the average of the three.

In [1]:
TRIO = [(2011, 0.5, -1.2, 0.7), (2012, -2.3, 4.4, -2.1), (2013, -2.6, -2.7, 5.3), (2014, 14.4, 24.8, -39.2), (2015, -4.7, 0.3, 4.4), (2016, 9.6, -36.1, 26.5), (2017, -3.9, 1.0, 2.9), (2018, -4.9, 3.0, 1.9), (2019, -5.6, -2.8, 8.5), (2020, -2.0, 4.6, -2.7), (2021, -3.2, -0.2, 3.4), (2022, -0.5, 0.9, -0.4), (2023, -9.3, 3.7, 5.5), (2024, -4.6, 0.2, 4.4), (2025, -7.7, 5.0, 2.7)]
print('Relative tracking difference, S&P 500 ETF trio (bp vs the family average)')
header = 'year'.rjust(6) + 'SPY'.rjust(9) + 'IVV'.rjust(9) + 'VOO'.rjust(9)
print(header + '   best that year')
for year, spy, ivv, voo in TRIO:
    row = {'SPY': spy, 'IVV': ivv, 'VOO': voo}
    best = max(row, key=row.get)
    line = str(year).rjust(6)
    for v in (spy, ivv, voo):
        line += format(v, '+.1f').rjust(9)
    print(line + '   ' + best)


Relative tracking difference, S&P 500 ETF trio (bp vs the family average)
  year      SPY      IVV      VOO   best that year
  2011     +0.5     -1.2     +0.7   VOO
  2012     -2.3     +4.4     -2.1   IVV
  2013     -2.6     -2.7     +5.3   VOO
  2014    +14.4    +24.8    -39.2   IVV
  2015     -4.7     +0.3     +4.4   VOO
  2016     +9.6    -36.1    +26.5   VOO
  2017     -3.9     +1.0     +2.9   VOO
  2018     -4.9     +3.0     +1.9   IVV
  2019     -5.6     -2.8     +8.5   VOO
  2020     -2.0     +4.6     -2.7   IVV
  2021     -3.2     -0.2     +3.4   VOO
  2022     -0.5     +0.9     -0.4   IVV
  2023     -9.3     +3.7     +5.5   VOO
  2024     -4.6     +0.2     +4.4   VOO
  2025     -7.7     +5.0     +2.7   IVV


## 2. Look at 2014 and 2016 before believing any of it

VOO 'lost' **39 bp** in 2014 and IVV 'lost' **36 bp** in 2016 — and both were back to normal the very next year. Those are not 40 basis points of real slippage; they are artefacts of how the data vendor time-stamps a dividend.

That matters enormously. The year-to-year wobble in this table has a standard deviation of **10.7 bp**, while the actual fee gap the trio contains is only **6.45 bp** (SPY charges 9.45, IVV and VOO charge 3). **The thing we are trying to see is smaller than the measurement error.** Whatever last year's ranking says, most of it is noise.

> 🔬 **For the quants** — the identification problem in one line: the estimator's per-year standard error exceeds the cross-sectional spread of the parameter. No amount of cleverness recovers a 6 bp ladder from a 10 bp floor in a single annual observation.

## 3. So does last year's winner repeat?

Rank the three ETFs each year and ask how strongly the ranking carries over. It barely does: the average year-over-year rank correlation is **+0.071** across 14 year-pairs — indistinguishable from a coin flip (*t* = +0.43; shuffling the years into a random order reproduces it 96% of the time).

And the rule built on it earns exactly what you would expect.

In [2]:
win_cheap, win_cheap_t, win_cheap_pos = (0.33, 0.1, '7/14')
cheap_lead, cheap_lead_t, cheap_lead_pos = (3.27, 1.26, '12/14')
switches, live_years = (10, 14)
print('S&P 500 ETF trio — annual rebalance, one-day lag, 1 bp one-way cost')
print()
print('  hold last year\'s winner  vs  hold the cheapest fund : '
      + format(win_cheap, '+.2f') + ' bp/yr  (t = ' + format(win_cheap_t, '+.2f')
      + ', positive in ' + win_cheap_pos + ' years)')
print('  hold the cheapest fund   vs  hold the flagship     : '
      + format(cheap_lead, '+.2f') + ' bp/yr  (t = ' + format(cheap_lead_t, '+.2f')
      + ', positive in ' + cheap_lead_pos + ' years)')
print()
print('  the winner rule changed fund ' + str(switches) + ' times in '
      + str(live_years) + ' years, to buy essentially nothing.')
print('  the cheapest rule never switched fund at all.')


S&P 500 ETF trio — annual rebalance, one-day lag, 1 bp one-way cost

  hold last year's winner  vs  hold the cheapest fund : +0.33 bp/yr  (t = +0.10, positive in 7/14 years)
  hold the cheapest fund   vs  hold the flagship     : +3.27 bp/yr  (t = +1.26, positive in 12/14 years)

  the winner rule changed fund 10 times in 14 years, to buy essentially nothing.
  the cheapest rule never switched fund at all.


## 4. Widen the fee gap and something *does* appear

Add three index **mutual funds** — Vanguard's VFIAX (4 bp), Fidelity's FXAIX (1.5 bp) and Schwab's SWPPX (2 bp) — and the ladder becomes wide enough that averaging over thirteen years pulls it out of the noise. Now the ranking persists (+0.253, *t* = +2.34), and the cheapest fund beats the flagship by **+10.64 bp a year** (*t* = +4.85, positive in 11/13 years).

The Nasdaq pair makes the same point with only two funds: **QQQM (15 bp) beat QQQ (20 bp) in 5/5 complete years**, by +8.55 bp a year on average (+8.79 over the four years a rule could actually have traded it — the first year always goes to forming the ranking).

**Two honest deductions from that +10.64.** It is measured with the fee sheet published *today*: Fidelity only cut FXAIX to 1.5 bp in 2019, so 'the cheapest fund' is partly chosen after the fact. Buy *all five* non-flagship funds equally instead — no fee sheet, no ranking, no forecast — and the gap is +5.21 bp/yr (*t* = +2.68), about half. And the cheapest-vs-flagship number is a *level*, not a memory: shuffle the calendar years into a random order and the same 'persistence' survives — because what persists is not last year's *result*, it is the fee, a fixed number printed on the fund's own website, in advance, for free. You never needed last year's tape.

## 5. The rule that works is not a strategy

The honest summary is: **buy a cheap share class, and then never touch it.** That is worth 3.3 to 5.2 bp a year with zero turnover — genuinely real, and genuinely a purchase decision rather than an edge. (The 10.6 bp version needs you to know which fund would end up cheapest; 5.2 bp is what you could have had knowing only that SPY was the dearest.)

It will also not justify moving money you already have, because switching an appreciated holding realises the capital gain. At the published 6.45 bp SPY→VOO gap, here is how long the cheaper fee takes to repay that one-off tax bill.

In [3]:
TAX = [(10, 23, 31, 37), (25, 58, 78, 92), (50, 116, 155, 185), (100, 233, 310, 369)]
print('Years for the 6.45 bp/yr fee gap to repay the tax on a SPY -> VOO switch')
print('(an ASSUMPTION grid — neither the gain nor the tax rate is on the tape)')
print()
print('embedded gain'.rjust(14) + 'sheltered'.rjust(12) + '15%'.rjust(8)
      + '20%'.rjust(8) + '23.8%'.rjust(8))
for gain, t15, t20, t238 in TAX:
    line = (str(gain) + '%').rjust(14) + '0'.rjust(12)
    for v in (t15, t20, t238):
        line += str(v).rjust(8)
    print(line)
print()
print('In a tax-sheltered account the switch is free.')
print('In a taxable one with a +50% gain at 20%, it takes 155 years to pay for itself.')


Years for the 6.45 bp/yr fee gap to repay the tax on a SPY -> VOO switch
(an ASSUMPTION grid — neither the gain nor the tax rate is on the tape)

 embedded gain   sheltered     15%     20%   23.8%
           10%           0      23      31      37
           25%           0      58      78      92
           50%           0     116     155     185
          100%           0     233     310     369

In a tax-sheltered account the switch is free.
In a taxable one with a +50% gain at 20%, it takes 155 years to pay for itself.


## 6. Live check — the machinery is not broken (offline **synthetic** data)

Before believing a null result, check that the detector fires when there really is something to find. The cell below runs on a **synthetic** panel of index funds, not the real tape: one world has a genuine 30 bp fee ladder, the other has identical fees. The pipeline must find persistence in the first and none in the second.

In [4]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))
import numpy as np
from td_persist import data, strategy as st
for label, ss in [('planted 30 bp fee ladder', 1.0), ('flat-fee null           ', 0.0)]:
    rho, gap = [], []
    for s in range(8):
        px, truth = data.synthetic_panel(signal_strength=ss, seed=913 + s)
        d = st.synthetic_detect(px, truth)
        rho.append(d['mean_spearman'])
        gap.append(d['cheapest_minus_leader_bp'])
    print(label + ' (SYNTHETIC, 8 seeds): rank persistence '
          + format(np.mean(rho), '+.3f') + '  |  cheapest-minus-dearest '
          + format(np.mean(gap), '+.2f') + ' bp/yr')


planted 30 bp fee ladder (SYNTHETIC, 8 seeds): rank persistence +0.725  |  cheapest-minus-dearest +35.66 bp/yr


flat-fee null            (SYNTHETIC, 8 seeds): rank persistence -0.079  |  cheapest-minus-dearest -0.29 bp/yr


## Verdict

- **Signal — Mixed.** The claim as posed fails exactly where it would be useful: among near-identical S&P 500 ETFs, last year's tracking-difference rank says nothing about next year's (+0.071, *t* = +0.43), because a 6.45 bp fee gap cannot be read through a 10.7 bp measurement floor. What *is* real is the thing underneath: across a wider ladder the cheapest fund beats the flagship by +10.64 bp/yr (*t* = +4.85) — +5.21 (*t* = +2.68) once you are not allowed to pick the cheapest fund with hindsight — and QQQM beat QQQ in 5/5 years. But that is a published *level*, not a memory — the ranking is the fee sheet, available in advance.
- **Tradability — Fragile.** The bankable version is a one-off purchase decision worth a few basis points with no turnover. The rotation version earns +0.33 bp/yr for 10 trades in 14 years, turns negative at any realistic switching cost, and is far too thin to move an existing taxable holding.